In [ ]:
import sys
import os
import time
import requests
import pandas as pd
from datasets import Dataset

# Ragas & Langchain (Ragas uses Langchain wrappers internally)
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import answer_relevancy, answer_correctness

# LlamaIndex for Naive RAG baseline
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from qdrant_client import QdrantClient

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from src import config

print("1. Initializing Evaluator Models...")
eval_llm = ChatGroq(model="llama3-70b-8192", api_key=config.GROQ_API_KEY, temperature=0)
eval_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}
)

ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)

print("2. Connecting to Naive RAG Baseline (Local Qdrant)...")
qdrant_client = QdrantClient(url=config.QDRANT_URL)
vector_store = QdrantVectorStore(client=qdrant_client, collection_name="sec_10k_filings")
naive_index = VectorStoreIndex.from_vector_store(vector_store, embed_model=eval_embeddings)
naive_query_engine = naive_index.as_query_engine(llm=None) # We just want retrieval, but we'll let it use default

print("3. Preparing Financial Multi-Hop Dataset...")
# 20 Complex Financial Questions (We will test a subset to avoid free-tier rate limits)
qa_pairs = [
    ("What major acquisitions did Microsoft make recently, and how did they impact the gaming division?", 
     "Microsoft acquired Activision Blizzard, which significantly expanded its gaming division, adding major franchises and boosting Xbox content and services revenue."),
    ("How does Apple rely on third-party manufacturing, and what risks are associated with it?", 
     "Apple relies heavily on outsourcing manufacturing to partners primarily in Asia. Risks include supply chain disruptions, geopolitical tensions, and reliance on single-source suppliers."),
    ("What are Microsoft's primary revenue streams within its Intelligent Cloud segment?", 
     "The Intelligent Cloud segment revenue is primarily driven by Azure and other cloud services, server products, and enterprise services."),
    ("Who are Apple's main competitors in the smartphone and wearables market?", 
     "Apple competes against other global tech companies like Samsung and Google in smartphones, and various fitness and smart device makers in wearables."),
    ("Describe the regulatory risks Microsoft faces regarding data privacy and AI.", 
     "Microsoft faces complex regulatory risks including GDPR in Europe, antitrust scrutiny over its AI partnerships, and emerging AI safety regulations globally.")
]

# Extract lists
questions = [item[0] for item in qa_pairs]
ground_truths = [item[1] for item in qa_pairs]

naive_answers = []
hybrid_answers = []

print("\n4. Running Queries against Naive RAG & Hybrid GraphRAG API...")
API_URL = "http://localhost:8000/ask"

for i, q in enumerate(questions):
    print(f"\nProcessing Q{i+1}: {q}")
    
    # --- A. Naive RAG (Vector Only) ---
    try:
        naive_response = naive_query_engine.query(q)
        naive_answers.append(str(naive_response))
    except Exception as e:
        naive_answers.append("Error retrieving context.")
    
    time.sleep(2) # Rate limit protection
    
    # --- B. Hybrid GraphRAG (Your Agent API) ---
    try:
        api_response = requests.post(API_URL, json={"question": q})
        if api_response.status_code == 200:
            hybrid_answers.append(api_response.json().get("answer", ""))
        else:
            hybrid_answers.append("API Error")
    except Exception as e:
        hybrid_answers.append("API Connection Error")
    
    time.sleep(3) # Rate limit protection

print("\n5. Running Ragas Evaluation (This takes a moment)...")

# Prepare Datasets
naive_data = Dataset.from_dict({"question": questions, "answer": naive_answers, "ground_truth": ground_truths})
hybrid_data = Dataset.from_dict({"question": questions, "answer": hybrid_answers, "ground_truth": ground_truths})

# Evaluate Naive
naive_results = evaluate(naive_data, metrics=[answer_relevancy, answer_correctness], llm=ragas_llm, embeddings=ragas_emb)
# Evaluate Hybrid
hybrid_results = evaluate(hybrid_data, metrics=[answer_relevancy, answer_correctness], llm=ragas_llm, embeddings=ragas_emb)

# Combine into a DataFrame
df_naive = naive_results.to_pandas()
df_hybrid = hybrid_results.to_pandas()

comparison = pd.DataFrame({
    "Metric": ["Answer Relevancy", "Answer Correctness (Accuracy)"],
    "Naive RAG (Vector Only)": [df_naive['answer_relevancy'].mean(), df_naive['answer_correctness'].mean()],
    "Nexus GraphRAG (Hybrid)": [df_hybrid['answer_relevancy'].mean(), df_hybrid['answer_correctness'].mean()]
})

print("\n\n🏆 FINAL COMPARISON RESULTS 🏆")
print(comparison.to_markdown(index=False))